# Regression metrics: MAE, MSE, RMSE, R², adjusted R²

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("data/placement.csv")
X_train, X_test, y_train, y_test = train_test_split(df[["cgpa"]], df["package"], test_size=0.2, random_state=2)
lr = LinearRegression().fit(X_train, y_train)
y_pred = lr.predict(X_test)

## The four metrics with scikit-learn

In [3]:
print("MAE ", mean_absolute_error(y_test, y_pred))
print("MSE ", mean_squared_error(y_test, y_pred))
print("RMSE", root_mean_squared_error(y_test, y_pred))
print("R2  ", r2_score(y_test, y_pred))

MAE  0.2884710931878175
MSE  0.12129235313495527
RMSE 0.34827051717731616
R2   0.780730147510384


## The same by hand

In [4]:
err = y_test.to_numpy() - y_pred
mae = np.abs(err).mean()
mse = (err ** 2).mean()
rmse = np.sqrt(mse)
ss_res = (err ** 2).sum()                                   # squared errors of the line
ss_tot = ((y_test - y_test.mean()) ** 2).sum()              # squared errors of 'guess the mean'
r2 = 1 - ss_res / ss_tot
print(mae, mse, rmse, ss_res, ss_tot, r2)

0.2884710931878175 0.12129235313495527 0.34827051717731616 4.851694125398211 22.12659 0.780730147510384


## Adjusted R²

In [5]:
def adjusted_r2(r2, n, k):
    # n rows, k input columns
    return 1 - (1 - r2) * (n - 1) / (n - 1 - k)

adjusted_r2(r2, n=len(y_test), k=1)

np.float64(0.7749598882343415)

## Adding useless random columns (averaged over 200 draws of the random numbers)

In [6]:
# the same 80/20 split each time; only the random numbers change; results are averaged over 200 draws
ks = [0, 1, 5, 10, 20]
total = np.zeros((len(ks), 4))
for seed in range(200):
    noise = np.random.default_rng(seed).random((len(df), 20))
    X_all = np.c_[df.cgpa.to_numpy(), noise]
    Xtr, Xte, ytr, yte = train_test_split(X_all, df.package.to_numpy(), test_size=0.2, random_state=2)
    for j, k in enumerate(ks):
        m = LinearRegression().fit(Xtr[:, :1 + k], ytr)
        r_tr = r2_score(ytr, m.predict(Xtr[:, :1 + k]))
        r_te = r2_score(yte, m.predict(Xte[:, :1 + k]))
        total[j] += [r_tr, adjusted_r2(r_tr, len(ytr), 1 + k), r_te, adjusted_r2(r_te, len(yte), 1 + k)]
pd.DataFrame(total / 200, index=pd.Index(ks, name="random cols"),
             columns=["R2 train", "adj R2 train", "R2 test", "adj R2 test"]).round(3)

,R2 train,adj R2 train,R2 test,adj R2 test
random cols,,,,
0,0.773,0.772,0.781,0.775
1,0.775,0.772,0.779,0.767
5,0.780,0.772,0.775,0.734
10,0.787,0.771,0.769,0.678
20,0.802,0.772,0.755,0.468
